## Self attention

In [1]:
import torch
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [2]:
attention_scores = inputs @ inputs.T
print(attention_scores)
print(attention_scores.shape)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
torch.Size([6, 6])


In [3]:
attention_weights = torch.softmax(attention_scores,dim=1)
print(attention_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [4]:
context_vector = attention_weights @ inputs
print(context_vector)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


# W_v, W_q, W_k matrices 
Using W_v, W_q, W_k as learnable parameters for context vector creation

In [14]:
import torch
import torch.nn as nn

class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias = False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias= qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias= qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias= qkv_bias)
    def forward(self, x):
        query = self.W_query(x)
        key = self.W_key(x)
        value = self.W_value(x)
        attention_scores = query @ key.T
        attention_weights = torch.softmax((attention_scores/(key.shape[-1]**0.5)), dim=-1)
        context_vectors = attention_weights @ value
        return context_vectors

In [15]:
self_attention = SelfAttention(d_in = 3, d_out = 2)
context_vectors = self_attention(inputs)
print(context_vectors)

tensor([[ 0.0366, -0.5689],
        [ 0.0374, -0.5524],
        [ 0.0374, -0.5526],
        [ 0.0373, -0.5524],
        [ 0.0373, -0.5565],
        [ 0.0374, -0.5511]], grad_fn=<MmBackward0>)


# Causal Attention


In [22]:
context_length = 6
keys_dim = 6
mask = torch.triu(torch.ones(context_length, context_length), diagonal =1)
masked = attention_scores.masked_fill(mask.bool(), -torch.inf)
attn_weights = torch.softmax(masked / keys_dim**0.5, dim=1)
print(mask)
print(masked)
print(attn_weights)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])
tensor([[0.9995,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.9544, 1.4950,   -inf,   -inf,   -inf,   -inf],
        [0.9422, 1.4754, 1.4570,   -inf,   -inf,   -inf],
        [0.4753, 0.8434, 0.8296, 0.4937,   -inf,   -inf],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654,   -inf],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4450, 0.5550, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2876, 0.3575, 0.3549, 0.0000, 0.0000, 0.0000],
        [0.2312, 0.2687, 0.2672, 0.2329, 0.0000, 0.0000],
        [0.1900, 0.2104, 0.2111, 0.1817, 0.2068, 0.0000],
        [0.1559, 0.1878, 0.1858, 0.1575, 0.1358, 0.1772]])


## Dropout Scaling 
Each remaining output is scaled by 1/(1-p)

In [18]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.1)    #1
example = torch.ones(6, 6)      #2
print(dropout(example))


tensor([[1.1111, 1.1111, 1.1111, 1.1111, 1.1111, 1.1111],
        [1.1111, 1.1111, 0.0000, 1.1111, 1.1111, 1.1111],
        [1.1111, 1.1111, 1.1111, 1.1111, 1.1111, 1.1111],
        [1.1111, 1.1111, 1.1111, 1.1111, 1.1111, 1.1111],
        [1.1111, 1.1111, 0.0000, 1.1111, 1.1111, 1.1111],
        [1.1111, 1.1111, 1.1111, 1.1111, 1.1111, 1.1111]])
